# PopSign Phase 4: Production Threshold Exploration

This notebook presents the reproducible Phase 4 run against the exact pinned MPNet vectors uploaded in Phase 3. It compares a global cosine-score threshold with a **runner-up margin**, then checks the selected rule on an untouched holdout split.

The labels are still provisional. The threshold stays inactive until the PopSign ASL team reviews the proposed sense mappings and generated cases.

In [1]:
import json
from pathlib import Path
from IPython.display import HTML, SVG, display

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

RESULT_PATH = ROOT / 'data' / 'phase4_threshold_evaluation.json'
result = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
counts = result['case_counts']
print(
    f"Loaded {counts['total']} cases: {counts['source_derived']} source-derived "
    f"and {counts['llm_generated']} LLM-generated."
)

Loaded 576 cases: 498 source-derived and 78 LLM-generated.


## Candidate global rule

A match is accepted only when its top cosine score clears the score threshold **and** its lead over the runner-up clears the margin.

In [2]:
candidate = result['candidate_global_threshold']
holdout = result['holdout_metrics']
rows = [
    ('Accept threshold', f"{candidate['accept_threshold']:.2f}"),
    ('Runner-up margin', f"{candidate['margin_threshold']:.2f}"),
    ('Top contexts averaged', str(candidate['top_k_contexts'])),
    ('Holdout accepted precision', f"{holdout['accepted_precision']:.1%}"),
    ('Holdout positive coverage', f"{holdout['positive_coverage']:.1%}"),
    ('Holdout ambiguous rejection', f"{holdout['ambiguous_rejection_rate']:.1%}"),
    ('Active in Supabase', 'No'),
]
display(HTML(
    '<table><thead><tr><th>Setting / metric</th><th>Result</th></tr></thead><tbody>'
    + ''.join(f'<tr><td>{name}</td><td>{value}</td></tr>' for name, value in rows)
    + '</tbody></table>'
))

Setting / metric,Result
Accept threshold,0.35
Runner-up margin,0.17
Top contexts averaged,2
Holdout accepted precision,95.8%
Holdout positive coverage,40.2%
Holdout ambiguous rejection,90.3%
Active in Supabase,No


## Precision-versus-coverage trade-off

The chart below shows why the runner-up margin matters more than the raw top score for this dataset. A stricter margin reduces incorrect automatic selections, but it also sends more correct phrases to review.

![Holdout threshold trade-off](../data/phase4_threshold_tradeoff.svg)

In [3]:
# Run this cell in Jupyter for a full-size chart.
display(SVG(filename=str(ROOT / 'data' / 'phase4_threshold_tradeoff.svg')))

## Does one global rule work?

The conservative global candidate clears the **overall** 95% precision and 90% ambiguous-rejection targets, but only accepts 40.2% of correct holdout phrases. On the harder LLM-only holdout, it reaches 80.0% accepted precision, 22.2% positive coverage, and 83.3% ambiguous rejection. That is not strong enough to activate.

In [4]:
word_rows = []
for word, values in sorted(result['candidate_per_word_thresholds'].items()):
    metrics = values['holdout_metrics']
    word_rows.append((
        word,
        f"{values['margin_threshold']:.2f}",
        f"{metrics['accepted_precision']:.1%}",
        f"{metrics['positive_coverage']:.1%}",
        f"{metrics['ambiguous_rejection_rate']:.1%}",
    ))
headers = ('Word', 'Candidate margin', 'Holdout precision', 'Holdout coverage', 'Holdout ambiguous rejection')
display(HTML(
    '<table><thead><tr>' + ''.join(f'<th>{h}</th>' for h in headers) + '</tr></thead><tbody>'
    + ''.join('<tr>' + ''.join(f'<td>{value}</td>' for value in row) + '</tr>' for row in word_rows)
    + '</tbody></table>'
))

Word,Candidate margin,Holdout precision,Holdout coverage,Holdout ambiguous rejection
about,0.02,66.7%,66.7%,50.0%
can,0.10,75.0%,50.0%,50.0%
love,0.04,100.0%,33.3%,100.0%


## Decision

1. Keep **0.35 score + 0.17 runner-up margin + top 2 contexts** as an inactive conservative baseline.
2. Do **not** add per-word overrides yet; their tiny, unreviewed calibration sets overfit and perform worse on holdout cases.
3. Have the PopSign ASL team review the generated labels and the six candidate video-to-sense mappings.
4. Improve the reference data for variants with copied or nearly identical contexts, then rerun this exact notebook.

The important finding is that a high cosine score alone does not prove the correct meaning. The runner-up gap is useful, but the current source labels and contexts are not yet strong enough for automatic production activation.

## Reproduce the run

```bash
uv sync --extra ml --group dev
uv run python scripts/evaluate_production_thresholds.py
```

Query embeddings are checkpointed under the ignored `.popsign-state/phase4/` directory. Re-running resumes from those 78 cached MPNet vectors instead of generating them again.